# Notebook Python confusion matrix generation and json free-text-annotation

#### Import Packages

In [1]:
import pandas as pd
import json
import os

#### Standardization of annotations contained in TSV files

In [ ]:
def process_annotation_field(field_value):
    """
    Process annotation fields by:
    - converting to lowercase
    - removing excess whitespace
    - splitting on commas
    - returning a clean list of annotations
    """
    if pd.isna(field_value) or field_value == "" or field_value is None:
        return []

    # Convert to lowercase
    text = str(field_value).lower()

    # Normalize excessive whitespace
    text = " ".join(text.split())

    # Split by commas and strip whitespace
    annotations = [item.strip() for item in text.split(",")]

    # Remove empty stringsSingle_Cell_gold_standard_consensus
    annotations = [item for item in annotations if item]

    return annotations

#### Load tsv file, creat json and load json file

In [ ]:
def load_tsv_and_create_json(tsv_file, json_file, discipline):
    """
    Load TSV and create JSON including all sources
    """
    df = pd.read_csv(tsv_file, sep="\t", encoding="utf-8")
    df.columns = df.columns.str.strip()

    all_tools = {}
    counter = 1

    for _, row in df.iterrows():
        tool_key = f"tools#{counter}"
        source = row.get("Source", "").strip()

        tool_data = {
            "name": row.get("Tool name", "").strip(),
            "discipline": discipline,
            "source": source,
            "topic": process_annotation_field(row.get("Topic", "")),
            "operation": process_annotation_field(row.get("Operation", "")),
            "input data": process_annotation_field(row.get("Input data", "")),
            "output data": process_annotation_field(row.get("Output data", "")),
            "input format": process_annotation_field(row.get("Input format", "")),
            "output format": process_annotation_field(row.get("Output format", "")),
        }

        all_tools[tool_key] = tool_data
        counter += 1

    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_tools, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_tools)} tools to {json_file}")


def load_json(json_file):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return None

#### Calculates confusion matrices for the LLM 

In [ ]:
def calculate_metrics_per_tool_annotation(json_data):
    """
    For each tool and each annotation type, calculate TP, FP, FN, TN, precision, recall, F1
    using lowercase cleaned annotations from process_annotation_field().
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    tool_names = set(entry["name"] for entry in json_data.values())

    expected_prov = {
        "consensus expert llm",
        "proposition deepseek-v3.1",
        "consensus expert",
    }

    for tool in tool_names:
        # initialize empty sets per provenance
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        # fill annotation sets
        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()

            if prov not in prov_sets:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # compute confusion metrics for each annotation type
        for field in annotation_fields:
            final_consensus_set = prov_sets["consensus expert llm"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]
            expert_set = prov_sets["consensus expert"][field]

            # missing by deepseek
            missing_in_deepseek = final_consensus_set - deepseek_set

            # FN = elements in LLM not predicted by DeepSeek but present in Expert
            FN_set = {x for x in missing_in_deepseek if x in expert_set}

            # mixed contributions = predicted by both even if not matched DeepSeek
            mixed_tp = {x for x in missing_in_deepseek if x not in expert_set}

            # TP: direct intersection + mixed tp
            TP_set = (final_consensus_set & deepseek_set) | mixed_tp

            # FP: predicted by deepseek but not in LLM
            FP_set = deepseek_set - final_consensus_set

            TN_set = set()  # always empty

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = len(TN_set)  # always 0

            precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
            recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
            f1_score = round(
                (
                    2 * precision * recall / (precision + recall)
                    if (precision + recall)
                    else 0
                ),
                2,
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),
            })

    return results


#### Calculates confusion matrices for the experts

In [ ]:
def calculate_metrics_expert_contribution(json_data):
    """
    For each tool and annotation type, calculate TP, FP, FN, TN, precision, recall, F1
    to measure the contribution of the expert consensus.
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    tool_names = set(entry["name"] for entry in json_data.values())

    expected_prov = {
        "consensus expert llm",
        "proposition deepseek-v3.1",
        "consensus expert",
    }

    for tool in tool_names:
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()
            if prov not in expected_prov:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # compute metrics per annotation field
        for field in annotation_fields:
            final_consensus_set = prov_sets["consensus expert llm"][field]
            expert_set = prov_sets["consensus expert"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]

            # elements in final consensus not predicted by expert
            missing_in_expert = final_consensus_set - expert_set

            # FN: missed by expert and not covered by DeepSeek
            FN_set = {x for x in missing_in_expert if x not in deepseek_set}

            # mixed contribution: missed by expert but covered by DeepSeek
            mixed_tp = {x for x in missing_in_expert if x in deepseek_set}

            # TP: direct expert hits + mixed contribution
            TP_set = (final_consensus_set & expert_set) | mixed_tp

            # FP: predicted by expert but not present in final consensus
            FP_set = expert_set - final_consensus_set

            TN_set = set()  # always empty

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = len(TN_set)  # always 0

            precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
            recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
            f1_score = round(
                (
                    2 * precision * recall / (precision + recall)
                    if (precision + recall)
                    else 0
                ),
                2,
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),
            })

    return results


#### Pipeline to generate JSON files, calculate and save a confusion matrix associated

In [ ]:
def save_metrics_to_tsv(metrics, tsv_file):
    """
    Save metrics dictionary/list to a TSV file.
    """
    df = pd.DataFrame(metrics)
    df.to_csv(tsv_file, sep="\t", index=False)
    print(f"Saved confusion metrics to {tsv_file}")


def run_pipeline_all_domains(
    raw_base="Raw_free_text_annotation",
    output_base="Confusion_matrix_free_text_annotation"
):
    """
    Run full pipeline (LLM + expert) for all domains
    - Read TSV from Raw_free_text_annotation
    - Save JSON in Raw folder
    - Save metrics in Confusion_matrix_free_text_annotation
    """

    domains = [
        "Metagenomic",
        "Systems_biology",
        "Single_Cell",
        "Phylogeny",
        "Neuroimaging",
        "Genetic_variant",
        "Microscopy",
    ]

    for domain in domains:
        print(f"\nProcessing domain: {domain}")

        # INPUT (RAW DATA)
        raw_domain_folder = os.path.join(raw_base, domain)

        tsv_file = os.path.join(
            raw_domain_folder,
            f"{domain}_gold_standard_consensus.tsv"
        )

        json_file = os.path.join(
            raw_domain_folder,
            f"{domain}_tools_annotations.json"
        )

        # OUTPUT (METRICS)
        output_domain_folder = os.path.join(output_base, domain)

        os.makedirs(output_domain_folder, exist_ok=True)

        # STEP 1: TSV to JSON
        load_tsv_and_create_json(
            tsv_file=tsv_file,
            json_file=json_file,
            discipline=domain
        )

        # STEP 2: LOAD JSON
        data = load_json(json_file)
        if not data:
            continue

        # STEP 3: LLM METRICS
        metrics_llm = calculate_metrics_per_tool_annotation(data)

        save_metrics_to_tsv(
            metrics_llm,
            tsv_file=os.path.join(
                output_domain_folder,
                f"{domain}_confusion_metrics_LLM.tsv"
            )
        )

        # STEP 4: EXPERT METRICS
        metrics_expert = calculate_metrics_expert_contribution(data)

        save_metrics_to_tsv(
            metrics_expert,
            tsv_file=os.path.join(
                output_domain_folder,
                f"{domain}_confusion_metrics_expert.tsv"
            )
        )

#### Run all the pipeline

In [8]:
run_pipeline_all_domains()


Processing domain: Metagenomic
Saved 36 tools to Raw_free_text_annotation/Metagenomic/Metagenomic_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Metagenomic/Metagenomic_confusion_metrics_LLM.tsv
Saved confusion metrics to Confusion_matrix_free_text_annotation/Metagenomic/Metagenomic_confusion_metrics_expert.tsv

Processing domain: Systems_biology
Saved 37 tools to Raw_free_text_annotation/Systems_biology/Systems_biology_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Systems_biology/Systems_biology_confusion_metrics_LLM.tsv
Saved confusion metrics to Confusion_matrix_free_text_annotation/Systems_biology/Systems_biology_confusion_metrics_expert.tsv

Processing domain: Single_Cell
Saved 15 tools to Raw_free_text_annotation/Single_Cell/Single_Cell_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Single_Cell/Single_Cell_confusion_metrics_LLM.tsv
Saved confusion metrics t

# Mapping with EDAM concepts 

In [ ]:
# from the consensus json
json_file = "Genetic_variant_tools_annotations.json"
data = {}
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# enumerate all annotation fields
annotation_fields = [
    "Topic",
    "Operation",
    "Input data",
    "Output data",
    "Input format",
    "Output format",
]

all_annotations = set()
all_topics = set ()
all_op = set()
all_data = set()
all_formats = set()


for tool_key, tool_data in data.items():
    if tool_data["source"] == "Consensus expert LLM":
        topics = tool_data.get("topic", [])
        operations = tool_data.get("operation", [])
        input_data = tool_data.get("input data", [])
        output_data = tool_data.get("output data", [])
        input_format = tool_data.get("input format", [])
        output_format = tool_data.get("output format", [])
        all_topics.update(set(topics))
        all_op.update(set(operations))
        all_data.update(set(input_data))
        all_data.update(set(output_data))
        all_formats.update(set(input_format))
        all_formats.update(set(output_format))

all_annotations = sorted(all_annotations)
all_topics  = sorted(all_topics)
pd.DataFrame(all_topics).to_csv("Genetic_variant_topics.csv")
all_op = sorted(all_op)
pd.DataFrame(all_op).to_csv("Genetic_variant_operations.csv")
all_data  = sorted(all_data)
pd.DataFrame(all_data).to_csv("Genetic_variant_data.csv")
all_formats = sorted(all_formats)
pd.DataFrame(all_formats).to_csv("Genetic_variant_formats.csv")


Total unique annotations found: 0
All unique annotations across all tools and fields:


## Count annotation 

In [ ]:
import pandas as pd
import os

# List of domains
domains = ["Metagenomic", "Neuroimaging", "Phylogeny", "Single_Cell", "Systems_biology", "Genetic_variant", "Microscopy"]

# Types of CSV files for each domain
file_types = ["topics", "operations", "data", "formats"]

# Create empty DataFrame to store counts
df_counts = pd.DataFrame(index=domains, columns=file_types)

# Function to count annotations in a CSV file (skip first line)
def count_annotations(file_path):
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, header=None, skiprows=1)  # skip first line
        # Count all non-empty rows
        count = df.dropna(how="all").shape[0]
        return count
    else:
        return 0  # File missing

# Populate DataFrame with counts per domain
for domain in domains:
    for file_type in file_types:
        file_name = f"{domain}_{file_type}.csv"
        df_counts.at[domain, file_type] = count_annotations(file_name)

# Convert counts to integers
df_counts = df_counts.astype(int)

# Add a "Total" row with global counts for each annotation type
df_counts.loc["Total"] = df_counts.sum(axis=0)

# Add new column: Number of tools
df_counts["Number of tools"] = [8, 5, 3, 3, 11, 11, 5, 45]

# Show the DataFrame
print(df_counts)

# Optionally, save to CSV
df_counts.to_csv("annotation_counts_summary_with_total.csv")


                 topics  operations  data  formats  Number of tools
Metagenomic          15          25    24       10                8
Neuroimaging         10          10    17       26                5
Phylogeny             8           8     9        5                3
Single_Cell           4          14    20       11                3
Systems_biology      51          65    83       52               11
Genetic_variant      16          19    29       19               10
Total               104         141   182      123               40


In [8]:
df_counts

,topics,operations,data,formats
Metagenomic,15,25,24,10
Neuroimaging,10,10,17,26
Phylogeny,8,8,9,5
Single_Cell,4,14,20,11
Systems_biology,51,65,83,52
Genetic_variant,16,19,29,19
Total,104,141,182,123
